# MASIVE-ALS — Docking en Kaggle (GPU gratis, SIN subir datos a mano)

**Paralelo con Colab:** este notebook procesa los PARES, Colab los IMPARES.
(O cambia `SUBCONJUNTO = 'todos'` en la celda 5 para hacerlos todos aqui.)

**Como usar (3 pasos):**
1. Arriba a la derecha: **Settings** -> **Accelerator** -> **GPU T4 x2** (o P100).
   Y confirma que **Internet** esta **activado** (en Settings, abajo).
2. Pega este notebook: **File > Import Notebook** (o **New Notebook** y sube el .ipynb).
3. Ejecuta las celdas en orden. Los datos se descargan solos desde GitHub
   (repo publico masive-als-data), no hay que subir nada.

**Cuota gratis:** 30 horas de GPU a la semana. Los resultados son locales a la
sesion: descargar `/kaggle/working/masive_als/resultados/resultados_kaggle.csv`
antes de cerrar (celda 6 lo imprime entero como respaldo).

**Nota tecnica:** AutoDock Vina usa CPU (no GPU). La GPU de Kaggle acelera otras
cosas, pero lo que de verdad suma aqui es la maquina gratis extra corriendo en
paralelo. La velocidad real la da el procesador (~2 vCPU).

In [ ]:
# CELDA 1: Verificar GPU y CPU
!nvidia-smi
print()
!nproc
print('Listo: GPU + CPU OK')

In [ ]:
# CELDA 2: Preparar (el binario de Vina se descarga solo en la celda 5)
import sys
print('Python', sys.version)
print('Nota: no hace falta instalar nada, el docking usa el binario oficial de Vina.')

In [ ]:
# CELDA 3: Preparar carpetas y descargar los paquetes de datos (auto)
import os, glob, tarfile, shutil, urllib.request

WORK = '/kaggle/working/masive_als'
for sub in ['receptores', 'ligandos', 'resultados', 'checkpoint']:
    os.makedirs(WORK + '/' + sub, exist_ok=True)

URLS = [
    'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/colab_receptores_plano.tar.gz',
    'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/colab_ligandos50_plano.tar.gz',
]
for url in URLS:
    destino = '/kaggle/working/' + os.path.basename(url)
    print('Descargando:', os.path.basename(url), flush=True)
    urllib.request.urlretrieve(url, destino)
tars = sorted(glob.glob('/kaggle/working/*.tar.gz'))

def es_receptor(nombre):
    return nombre in ('TDP43.pdbqt', 'SOD1.pdbqt', 'FUS.pdbqt')

extract_dir = '/kaggle/working/tmp_extract'
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
os.makedirs(extract_dir)

moved_r = moved_l = 0
for tar in tars:
    print('Extrayendo:', os.path.basename(tar), flush=True)
    with tarfile.open(tar) as t:
        t.extractall(extract_dir)

# Copiar cada .pdbqt a su carpeta (aunque venga en subcarpetas)
for raiz, _, archivos in os.walk(extract_dir):
    for a in archivos:
        if a.endswith('.pdbqt'):
            destino = WORK + '/receptores/' + a if es_receptor(a) else WORK + '/ligandos/' + a
            shutil.copy(os.path.join(raiz, a), destino)
            if es_receptor(a):
                moved_r += 1
            else:
                moved_l += 1

print('Receptores: %d  |  Ligandos: %d' % (moved_r, moved_l), flush=True)
print('Receptores en carpeta:', sorted(os.listdir(WORK + '/receptores')), flush=True)
print('Ligandos en carpeta:', len(os.listdir(WORK + '/ligandos')), flush=True)
if moved_r + moved_l == 0:
    print('ERROR: no se pudieron obtener los paquetes de datos. Revisa que Internet este activado.')

In [ ]:
# CELDA 4: Definir receptores (coordenadas de la literatura)
import os
RECEPTORES = {
    'TDP43': {
        'archivo': WORK + '/receptores/TDP43.pdbqt',
        'centro': [28.3, 43.7, 52.5],
        'tamano': [25, 25, 25]
    },
    'SOD1': {
        'archivo': WORK + '/receptores/SOD1.pdbqt',
        'centro': [27.9, 111.8, 64.4],
        'tamano': [25, 25, 25]
    },
    'FUS': {
        'archivo': WORK + '/receptores/FUS.pdbqt',
        'centro': [-14.5, 15.1, -7.8],
        'tamano': [25, 25, 25]
    }
}
for nombre, info in RECEPTORES.items():
    ok = os.path.exists(info['archivo'])
    print(nombre, 'EXISTE' if ok else 'FALTA - ejecuta la celda 3')

In [ ]:
# CELDA 5: Docking completo (binario Vina, subproceso por par)
import csv, glob, os, shutil, subprocess, tarfile, time, urllib.request
WORK = '/kaggle/working/masive_als'
VINA_BIN = '/kaggle/working/vina_bin'

# --- subconjunto: 'pares', 'impares' o 'todos' ---
SUBCONJUNTO = 'pares'

# --- 1) Descargar el binario oficial de Vina ---
if not os.path.exists(VINA_BIN):
    url = 'https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64'
    print('Descargando binario Vina...', flush=True)
    urllib.request.urlretrieve(url, VINA_BIN)
    os.chmod(VINA_BIN, 0o755)
print('Binario Vina listo', flush=True)

# --- 2) Datos frescos (no depende de la celda 3) ---
for sub in ['receptores', 'ligandos', 'resultados']:
    os.makedirs(WORK + '/' + sub, exist_ok=True)
URLS = [
    'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/colab_receptores_plano.tar.gz',
    'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/colab_ligandos50_plano.tar.gz',
]
for url in URLS:
    dest = '/kaggle/working/' + os.path.basename(url)
    if os.path.exists(dest):
        os.remove(dest)
    print('Descargando:', os.path.basename(url), flush=True)
    urllib.request.urlretrieve(url, dest)
extract_dir = '/kaggle/working/tmp_extract'
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
os.makedirs(extract_dir)
for tar in sorted(glob.glob('/kaggle/working/*.tar.gz')):
    with tarfile.open(tar) as t:
        try:
            t.extractall(extract_dir, filter='data')
        except TypeError:
            t.extractall(extract_dir)
def es_receptor(n):
    return n in ('TDP43.pdbqt', 'SOD1.pdbqt', 'FUS.pdbqt')
for d in ('receptores', 'ligandos'):
    for f in glob.glob(WORK + '/' + d + '/*.pdbqt'):
        os.remove(f)
for raiz, _, archivos in os.walk(extract_dir):
    for a in archivos:
        if a.endswith('.pdbqt'):
            destino = WORK + '/receptores/' if es_receptor(a) else WORK + '/ligandos/'
            shutil.copy(os.path.join(raiz, a), destino + a)

# --- 2b) PDBQT reparados por Colab (sobreescriben los corruptos del tar) ---
REPARADOS = ['CHEMBL1076399', 'CHEMBL1163427', 'CHEMBL1203109', 'CHEMBL1203132',
             'CHEMBL1203140', 'CHEMBL1203155', 'CHEMBL1203199', 'CHEMBL1203224',
             'CHEMBL1203252', 'CHEMBL1204421', 'CHEMBL1207772', 'CHEMBL1208195',
             'CHEMBL152893']
for nombre in REPARADOS:
    try:
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/'
            'ligandos_reparados/%s.pdbqt' % nombre,
            WORK + '/ligandos/' + nombre + '.pdbqt')
    except Exception:
        pass  # si aun no estan en GitHub, se usa el del tar (puede fallar en vina)
n_reparados = 0
for nombre in REPARADOS:
    lig = WORK + '/ligandos/' + nombre + '.pdbqt'
    try:
        if os.path.exists(lig) and any(l.startswith(('ATOM', 'HETATM'))
                                       for l in open(lig, errors='replace')):
            n_reparados += 1
    except Exception:
        pass
print('PDBQT reparados aplicados:', n_reparados, flush=True)

print('Receptores:', len(glob.glob(WORK + '/receptores/*.pdbqt')),
      '| Ligandos:', len(glob.glob(WORK + '/ligandos/*.pdbqt')), flush=True)

# --- 3) Receptores ---
RECEPTORES = {
    'TDP43': {'archivo': WORK + '/receptores/TDP43.pdbqt', 'centro': [28.3, 43.7, 52.5], 'tamano': [25, 25, 25]},
    'SOD1': {'archivo': WORK + '/receptores/SOD1.pdbqt', 'centro': [27.9, 111.8, 64.4], 'tamano': [25, 25, 25]},
    'FUS': {'archivo': WORK + '/receptores/FUS.pdbqt', 'centro': [-14.5, 15.1, -7.8], 'tamano': [25, 25, 25]},
}

# --- 4) FUS.pdbqt plano (sin MODEL/ENDMDL) ---
fus = RECEPTORES['FUS']['archivo']
try:
    lines = open(fus).read().splitlines()
    ini = next((i for i, l in enumerate(lines) if l.startswith('MODEL')), None)
    fin = next((i for i, l in enumerate(lines) if l.strip() == 'ENDMDL'), None)
    if ini is not None and fin is not None and fin >= ini:
        keep = lines[ini + 1:fin]
    else:
        keep = lines
    if not any(l.startswith(('ATOM', 'HETATM')) for l in keep):
        keep = lines
    open(fus, 'w').write('\n'.join(keep) + '\n')
    print('FUS plano: ATOM=%d' % sum(1 for l in keep if l.startswith(('ATOM', 'HETATM'))), flush=True)
except Exception as ex:
    print('ERROR reparando FUS:', str(ex)[:100], flush=True)

# --- 5) CSV (reanuda si ya hay filas) ---
CSV = WORK + '/resultados/resultados_kaggle.csv'
if not os.path.exists(CSV):
    with open(CSV, 'w', newline='') as f:
        csv.writer(f).writerow(['ligand', 'target', 'energy', 'timestamp'])
SALTADOS = WORK + '/resultados/saltados.txt'
saltados = set()
if os.path.exists(SALTADOS):
    saltados = set(l.strip() for l in open(SALTADOS) if l.strip())

hechos = {}
for r in csv.DictReader(open(CSV)):
    hechos.setdefault(r['ligand'], set()).add(r['target'])

# --- 5b) Base de Colab: pares ya hechos (descargados de GitHub) ---
BASE = WORK + '/resultados/resultados_colab.csv'
try:
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/resultados_colab.csv', BASE)
    for r in csv.DictReader(open(BASE)):
        hechos.setdefault(r['ligand'], set()).add(r['target'])
    print('Base Colab fusionada: %d pares ya hechos' %
          sum(len(v) for v in hechos.values()), flush=True)
except Exception as ex:
    print('AVISO sin base Colab:', str(ex)[:80], flush=True)

ligs = sorted(glob.glob(WORK + '/ligandos/*.pdbqt'))
# --- aplicar subconjunto por indice ---
if SUBCONJUNTO in ('pares', 'impares'):
    paridad = 0 if SUBCONJUNTO == 'pares' else 1
    ligs = [l for i, l in enumerate(ligs) if i % 2 == paridad]
print('Ligandos (subconjunto=%s):' % SUBCONJUNTO, len(ligs), flush=True)

pendientes = []
for lig in ligs:
    nombre = os.path.basename(lig).replace('.pdbqt', '')
    if nombre in saltados:
        continue
    para = hechos.get(nombre, set())
    for target in RECEPTORES:
        if target not in para:
            pendientes.append((lig, nombre, target))
print('Pares pendientes:', len(pendientes), flush=True)

# --- 6) Acoplar cada par en un SUBPROCESO ---
def acoplar(lig, target):
    info = RECEPTORES[target]
    cmd = [VINA_BIN,
           '--receptor', info['archivo'],
           '--ligand', lig,
           '--center_x', str(info['centro'][0]),
           '--center_y', str(info['centro'][1]),
           '--center_z', str(info['centro'][2]),
           '--size_x', str(info['tamano'][0]),
           '--size_y', str(info['tamano'][1]),
           '--size_z', str(info['tamano'][2]),
           '--exhaustiveness', '2', '--num_modes', '3', '--seed', '42']
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
        out = (r.stdout or '') + (r.stderr or '')
        if r.returncode != 0:
            return None, 'vina rc=%d %s' % (r.returncode, out[-150:])
        for ln in out.splitlines():
            s = ln.split()
            if len(s) >= 2 and s[0] == '1':
                try:
                    return round(float(s[1]), 4), None
                except ValueError:
                    pass
        return None, 'sin afinidad: ' + out[-120:]
    except Exception as ex:
        return None, str(ex)[:100]

def tiene_atomos(lig):
    try:
        with open(lig, errors='replace') as f:
            return any(l.startswith(('ATOM', 'HETATM')) for l in f)
    except Exception:
        return False

if pendientes:
    t0 = time.time()
    n_ok = 0
    for lig, nombre, target in pendientes:
        if not tiene_atomos(lig):
            print('LIGANDO_VACIO', nombre, '- se salta', flush=True)
            with open(SALTADOS, 'a') as f:
                f.write(nombre + '\n')
            continue
        energia, err = acoplar(lig, target)
        if energia is not None:
            with open(CSV, 'a', newline='') as f:
                csv.writer(f).writerow([nombre, target, energia, time.strftime('%Y-%m-%d %H:%M:%S')])
            n_ok += 1
        else:
            print('ERROR', nombre, target, err, flush=True)
        if n_ok % 15 == 0 and n_ok > 0:
            print('[%d acoplados] %.1f min' % (n_ok, (time.time() - t0) / 60), flush=True)
    print('Acoplados ahora:', n_ok, flush=True)

print(flush=True)
print('=== TANDAS COMPLETADAS ===', flush=True)
print('Filas en CSV:', len(list(csv.DictReader(open(CSV)))), flush=True)

In [ ]:
# CELDA 6: Resumen de resultados (con respaldo completo en la celda)
import csv, glob, os
rows = list(csv.DictReader(open(CSV)))
print('Total resultados:', len(rows))
if rows:
    best = sorted(rows, key=lambda x: float(x['energy']))[:10]
    print()
    print('Top 10:')
    for r in best:
        print('  ', r['ligand'], r['target'], r['energy'])

print()
print('Descargar: panel derecho -> masive_als/resultados/resultados_kaggle.csv')
print()
print('RESULTADOS TOTALES (respaldo en celda):')
print(open(CSV).read())